In [ ]:
import pandas as pd
import requests
import io

# 1. Daten via OSF API laden
osf_project_id = "dm37b"  # Deine korrekte OSF-ID
api_url = f"https://api.osf.io/v2/nodes/{osf_project_id}/files/osfstorage/"

print("Verbinde mit OSF und suche nach Datensätzen...")
response = requests.get(api_url)
files_data = response.json()

dataframes = []

# Alle Dateien im OSF-Ordner durchgehen
for file in files_data.get('data', []):
    file_name = file['attributes']['name']
    
    # Wir wollen nur die .csv Dateien einlesen
    if file_name.endswith('.csv'):
        download_url = file['links']['download']
        file_content = requests.get(download_url).content
        
        # CSV in einen pandas DataFrame laden und zur Liste hinzufügen
        df = pd.read_csv(io.StringIO(file_content.decode('utf-8')))
        dataframes.append(df)

# Alle Datensätze zu einer großen Tabelle zusammenfügen
if len(dataframes) > 0:
    raw_data = pd.concat(dataframes, ignore_index=True)
    print(f"Erfolgreich {len(dataframes)} Datensätze geladen und zusammengefügt.")
    
    # 2. Preprocessing (Datenreinigung)
    clean_data = raw_data[raw_data['task'] == 'test_trial'].copy()
    clean_data['accuracy'] = clean_data['correct'].astype(int)
    clean_data['study_position'] = clean_data['study_position'].replace('N/A', -1)
    clean_data['study_position'] = pd.to_numeric(clean_data['study_position'])
    print(f"Datenreinigung abgeschlossen. Es verbleiben {len(clean_data)} Test-Trials.")
else:
    print("Keine CSV-Dateien gefunden. Ist das Projekt noch auf 'Private' gestellt?")

In [ ]:
# Unabhängige Variable: 'study_position' (1 bis 15)
# Abhängige Variable: 'accuracy' (0.0 bis 1.0)

# Nur die echten Lern-Wörter (Targets) betrachten
targets_data = clean_data[clean_data['condition'] == 'old'].copy()

# Aggregation auf Item-Ebene (für Diagramme)
position_accuracy = targets_data.groupby('study_position')['accuracy'].mean().reset_index()

# Aggregation auf Personen-Ebene (für statistische Tests)
subject_position_accuracy = targets_data.groupby(['subject', 'study_position'])['accuracy'].mean().reset_index()

mean_rt = targets_data['rt'].mean()
print("Daten erfolgreich aggregiert!")
print(f"Die durchschnittliche Reaktionszeit lag bei {mean_rt:.2f} Millisekunden.")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Tabelle erstellen
desc_stats = clean_data.groupby('condition')['accuracy'].agg(['mean', 'std', 'count']).reset_index()
desc_stats.rename(columns={'mean': 'Trefferrate (M)', 'std': 'Standardabweichung (SD)', 'count': 'Trials'}, inplace=True)
print("--- Deskriptive Statistik ---")
print(desc_stats)

# Diagramm zeichnen
sns.set_theme(style="whitegrid")
plt.figure(figsize=(10, 6))

sns.lineplot(data=position_accuracy, x='study_position', y='accuracy', 
             marker='o', markersize=8, linewidth=2.5, color='#2c3e50')

plt.title('Erinnerungsleistung in Abhängigkeit der Listenposition', fontsize=16, fontweight='bold', pad=15)
plt.xlabel('Position in der Lernliste (1 = Erstes Wort, 15 = Letztes Wort)', fontsize=12)
plt.ylabel('Trefferrate (Accuracy)', fontsize=12)
plt.ylim(0, 1.05)
plt.xticks(range(1, 16))
plt.tight_layout()
plt.show()

In [ ]:
import scipy.stats as stats

print("--- Prüfung der statistischen Voraussetzungen ---")

# Normalverteilung prüfen
stat_sw, p_sw = stats.shapiro(position_accuracy['accuracy'])
print("\n1. Normalverteilung (Shapiro-Wilk):")
if p_sw > 0.05:
    print(f"p = {p_sw:.3f} -> Alles im grünen Bereich (Normalverteilung angenommen).")
else:
    print(f"p = {p_sw:.3f} -> ACHTUNG: Die Daten weichen signifikant von einer Normalverteilung ab.")

# Varianzhomogenität prüfen (Anfang vs. Mitte)
primacy_data = subject_position_accuracy[subject_position_accuracy['study_position'].isin([1, 2, 3])]['accuracy']
middle_data = subject_position_accuracy[subject_position_accuracy['study_position'].isin([7, 8, 9])]['accuracy']

stat_lev, p_lev = stats.levene(primacy_data.dropna(), middle_data.dropna())
print("\n2. Varianzhomogenität (Levene-Test):")
if p_lev > 0.05:
    print(f"p = {p_lev:.3f} -> Alles im grünen Bereich (Gleiche Varianzen).")
else:
    print(f"p = {p_lev:.3f} -> ACHTUNG: Varianzen unterscheiden sich signifikant.")

In [ ]:
import numpy as np

def get_zone_accuracy(positions):
    zone_data = targets_data[targets_data['study_position'].isin(positions)]
    return zone_data.groupby('subject')['accuracy'].mean()

primacy_acc = get_zone_accuracy([1, 2, 3])
middle_acc = get_zone_accuracy([7, 8, 9])
recency_acc = get_zone_accuracy([13, 14, 15])

common_subjects = primacy_acc.index.intersection(middle_acc.index).intersection(recency_acc.index)
primacy_acc = primacy_acc[common_subjects]
middle_acc = middle_acc[common_subjects]
recency_acc = recency_acc[common_subjects]

# t-Tests
t_prim, p_prim = stats.ttest_rel(primacy_acc, middle_acc)
t_rec, p_rec = stats.ttest_rel(recency_acc, middle_acc)

# Effektstärke (Cohen's d)
def cohens_d(x, y):
    diff = x - y
    return np.mean(diff) / np.std(diff, ddof=1)

print("--- Ergebnisse der Gruppenvergleiche ---")
print(f"Primacy-Effekt (Anfang vs. Mitte): t = {t_prim:.2f}, p = {p_prim:.4f}, Cohen's d = {cohens_d(primacy_acc, middle_acc):.2f}")
print(f"Recency-Effekt (Ende vs. Mitte):   t = {t_rec:.2f}, p = {p_rec:.4f}, Cohen's d = {cohens_d(recency_acc, middle_acc):.2f}")

In [ ]:
import statsmodels.formula.api as smf
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

def assign_zone(pos):
    if pos in [1, 2, 3]: return '1_Primacy'
    elif pos in [7, 8, 9]: return '2_Middle'
    elif pos in [13, 14, 15]: return '3_Recency'
    else: return 'Other'

mlm_data = targets_data.copy()
mlm_data['zone'] = mlm_data['study_position'].apply(assign_zone)
mlm_data = mlm_data[mlm_data['zone'] != 'Other']

# --- NEU: Die Reihenfolge zwingend festlegen ---
kategorien = ['1_Primacy', '2_Middle', '3_Recency']
mlm_data['zone'] = pd.Categorical(mlm_data['zone'], categories=kategorien, ordered=True)

# Datensatz sicherheitshalber sortieren, damit die Linien sauber gezeichnet werden
mlm_data = mlm_data.sort_values(['subject', 'zone'])
# -----------------------------------------------

# Modell berechnen
model = smf.mixedlm("accuracy ~ C(zone)", mlm_data, groups=mlm_data["subject"])
result = model.fit()

# Vorhersagen für den Plot
mlm_data['predicted_accuracy'] = result.fittedvalues

# Plot
sns.set_theme(style="whitegrid")
plt.figure(figsize=(10, 6))

# Graue Linien: Jede individuelle Versuchsperson
sns.lineplot(data=mlm_data, x='zone', y='predicted_accuracy', hue='subject', 
             palette='Greys', linewidth=1.5, alpha=0.4, legend=False)

# Rote Linie: Durchschnittlicher wahrer Effekt
sns.lineplot(data=mlm_data, x='zone', y='accuracy', color='red', 
             linewidth=4, marker='o', markersize=10, label='Fixed Effect (Modellierter Durchschnitt)', errorbar=None)

plt.title('Multilevel-Modell: Individuelle vs. Durchschnittliche Leistung', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Zonen der Lernliste', fontsize=12)
plt.ylabel('Trefferrate (Accuracy)', fontsize=12)
plt.ylim(0, 1.05)
plt.legend()
plt.tight_layout()
plt.show()

# Zahlentabelle ausgeben
print(result.summary())